# Course 14: MLOps with Vertex AI - Model Evaluation
**GCP Machine Learning Engineer Certification - Sections 5 & 6**

This notebook covers hands-on exercises for model evaluation:
1. Classification Evaluation (sklearn metrics)
2. Confusion Matrix Visualization
3. ROC Curve & Precision-Recall Curve
4. Regression Metrics Calculation
5. LLM Evaluation: Simple Judge using Gemini
6. Vertex AI Model Evaluation SDK
7. Slice-Based Evaluation
8. A/B Test Significance Calculator

---
## 0. Setup & Dependencies

In [ ]:
# Install required packages (uncomment if needed)
# !pip install scikit-learn matplotlib numpy pandas scipy --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve,
    mean_absolute_error, mean_squared_error, r2_score,
)
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set dark theme for matplotlib to match the study guide
plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0b0d14',
    'axes.facecolor': '#10131e',
    'axes.edgecolor': '#232840',
    'text.color': '#e6e8f4',
    'axes.labelcolor': '#8e94b5',
    'xtick.color': '#8e94b5',
    'ytick.color': '#8e94b5',
    'grid.color': '#1a1f34',
    'font.size': 12,
})

print("All imports loaded successfully.")

---
## 1. Classification Evaluation: sklearn Metrics

Compute all major classification metrics on a sample binary classification problem.

In [ ]:
# Generate sample predictions for a binary classifier
# Simulating a fraud detection model (imbalanced: 5% fraud)
np.random.seed(42)
n_samples = 1000

# True labels (5% positive = fraud)
y_true = np.zeros(n_samples, dtype=int)
y_true[:50] = 1  # 50 fraud cases out of 1000
np.random.shuffle(y_true)

# Model prediction probabilities (simulated)
y_prob = np.where(
    y_true == 1,
    np.random.beta(5, 2, size=n_samples),   # fraud cases: higher scores
    np.random.beta(2, 8, size=n_samples),    # normal cases: lower scores
)

# Binary predictions at threshold 0.5
y_pred = (y_prob >= 0.5).astype(int)

print("=== Classification Metrics (threshold=0.5) ===")
print(f"Total samples: {n_samples} ({y_true.sum()} positive, {n_samples - y_true.sum()} negative)")
print(f"Class imbalance: {y_true.mean()*100:.1f}% positive\n")

print(f"Accuracy:      {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision:     {precision_score(y_true, y_pred):.4f}")
print(f"Recall:        {recall_score(y_true, y_pred):.4f}")
print(f"F1 Score:      {f1_score(y_true, y_pred):.4f}")
print(f"ROC-AUC:       {roc_auc_score(y_true, y_prob):.4f}")
print(f"PR-AUC:        {average_precision_score(y_true, y_prob):.4f}")

print("\n=== Full Classification Report ===")
print(classification_report(y_true, y_pred, target_names=["Normal", "Fraud"]))

In [ ]:
# Demonstrate the effect of threshold on precision/recall
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
print("=== Effect of Classification Threshold ===")
print(f"{'Threshold':>10} {'Precision':>10} {'Recall':>10} {'F1':>10} {'FP':>6} {'FN':>6}")
print("-" * 58)

for t in thresholds:
    pred = (y_prob >= t).astype(int)
    p = precision_score(y_true, pred, zero_division=0)
    r = recall_score(y_true, pred)
    f = f1_score(y_true, pred, zero_division=0)
    cm = confusion_matrix(y_true, pred)
    fp = cm[0, 1] if cm.shape[1] > 1 else 0
    fn = cm[1, 0]
    print(f"{t:>10.1f} {p:>10.4f} {r:>10.4f} {f:>10.4f} {fp:>6} {fn:>6}")

print("\nKey insight: Lower threshold = higher recall (catch more fraud) but more false positives")
print("Higher threshold = higher precision (fewer false alarms) but miss more fraud")

---
## 2. Confusion Matrix Visualization

In [ ]:
# Visualize confusion matrix
cm = confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
im1 = axes[0].imshow(cm, cmap='Blues', alpha=0.8)
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold', color='#e6e8f4')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(['Normal', 'Fraud'])
axes[0].set_yticklabels(['Normal', 'Fraud'])

for i in range(2):
    for j in range(2):
        color = '#0b0d14' if cm[i, j] > cm.max() / 2 else '#e6e8f4'
        label = f"{cm[i, j]}\n"
        if i == 0 and j == 0: label += "TN"
        elif i == 0 and j == 1: label += "FP"
        elif i == 1 and j == 0: label += "FN"
        else: label += "TP"
        axes[0].text(j, i, label, ha='center', va='center', fontsize=14, 
                     fontweight='bold', color=color)

# Normalized (percentages)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
im2 = axes[1].imshow(cm_norm, cmap='Blues', alpha=0.8, vmin=0, vmax=1)
axes[1].set_title('Confusion Matrix (Normalized)', fontweight='bold', color='#e6e8f4')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')
axes[1].set_xticks([0, 1])
axes[1].set_yticks([0, 1])
axes[1].set_xticklabels(['Normal', 'Fraud'])
axes[1].set_yticklabels(['Normal', 'Fraud'])

for i in range(2):
    for j in range(2):
        color = '#0b0d14' if cm_norm[i, j] > 0.5 else '#e6e8f4'
        axes[1].text(j, i, f"{cm_norm[i, j]:.2%}", ha='center', va='center', 
                     fontsize=14, fontweight='bold', color=color)

plt.tight_layout()
plt.show()

print(f"\nTrue Negatives (TN): {cm[0,0]} - Correctly identified as normal")
print(f"False Positives (FP): {cm[0,1]} - Normal flagged as fraud (false alarm)")
print(f"False Negatives (FN): {cm[1,0]} - Fraud missed (dangerous!)")
print(f"True Positives (TP): {cm[1,1]} - Correctly caught fraud")

---
## 3. ROC Curve and Precision-Recall Curve

In [ ]:
# ROC and Precision-Recall curves
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- ROC Curve ---
fpr, tpr, thresholds_roc = roc_curve(y_true, y_prob)
roc_auc = roc_auc_score(y_true, y_prob)

axes[0].plot(fpr, tpr, color='#3b82f6', linewidth=2.5, label=f'Model (AUC = {roc_auc:.3f})')
axes[0].plot([0, 1], [0, 1], color='#545a78', linestyle='--', linewidth=1, label='Random (AUC = 0.500)')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='#3b82f6')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve', fontweight='bold', color='#e6e8f4', fontsize=14)
axes[0].legend(loc='lower right', framealpha=0.3)
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1.02])
axes[0].grid(True, alpha=0.3)

# Add threshold annotation
idx_05 = np.argmin(np.abs(thresholds_roc - 0.5))
axes[0].scatter([fpr[idx_05]], [tpr[idx_05]], color='#ef4444', s=80, zorder=5)
axes[0].annotate(f't=0.5', (fpr[idx_05], tpr[idx_05]), textcoords="offset points",
                 xytext=(10, -15), color='#ef4444', fontsize=10)

# --- Precision-Recall Curve ---
precision, recall, thresholds_pr = precision_recall_curve(y_true, y_prob)
pr_auc = average_precision_score(y_true, y_prob)

axes[1].plot(recall, precision, color='#22c55e', linewidth=2.5, label=f'Model (AP = {pr_auc:.3f})')
axes[1].axhline(y=y_true.mean(), color='#545a78', linestyle='--', linewidth=1, 
                label=f'Baseline ({y_true.mean():.3f})')
axes[1].fill_between(recall, precision, alpha=0.1, color='#22c55e')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve', fontweight='bold', color='#e6e8f4', fontsize=14)
axes[1].legend(loc='upper right', framealpha=0.3)
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.02])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"ROC-AUC: {roc_auc:.4f} - Measures ranking ability across all thresholds")
print(f"PR-AUC:  {pr_auc:.4f} - Better metric for imbalanced data (5% positive rate)")
print(f"\nFor this imbalanced fraud dataset, PR-AUC is more informative than ROC-AUC.")

---
## 4. Regression Metrics Calculation

In [ ]:
# Generate sample regression data (house price prediction)
np.random.seed(42)
n = 200

# True house prices (in $1000s)
y_true_reg = np.random.normal(300, 100, n)
y_true_reg = np.clip(y_true_reg, 50, 800)  # Realistic range

# Model predictions (with some noise)
noise = np.random.normal(0, 30, n)
y_pred_reg = y_true_reg + noise

# Add some outlier predictions
outlier_idx = np.random.choice(n, 5, replace=False)
y_pred_reg[outlier_idx] += np.random.choice([-1, 1], 5) * 150  # Big errors

# Compute all regression metrics
mae = mean_absolute_error(y_true_reg, y_pred_reg)
mse = mean_squared_error(y_true_reg, y_pred_reg)
rmse = np.sqrt(mse)
r2 = r2_score(y_true_reg, y_pred_reg)
mape = np.mean(np.abs((y_true_reg - y_pred_reg) / y_true_reg)) * 100

print("=== Regression Metrics (House Price Prediction) ===")
print(f"MAE:   ${mae:.2f}K  - Average absolute error")
print(f"MSE:   {mse:.2f}    - Mean squared error (penalizes outliers)")
print(f"RMSE:  ${rmse:.2f}K  - Root MSE (same units as target)")
print(f"R^2:   {r2:.4f}     - Proportion of variance explained")
print(f"MAPE:  {mape:.2f}%   - Mean absolute percentage error")

print(f"\nInterpretation:")
print(f"  - Our model's predictions are off by ${mae:.0f}K on average")
print(f"  - The model explains {r2*100:.1f}% of the variance in house prices")
print(f"  - Predictions are off by {mape:.1f}% on average")

# Visualize actual vs predicted
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter plot
axes[0].scatter(y_true_reg, y_pred_reg, alpha=0.5, color='#3b82f6', s=20)
axes[0].scatter(y_true_reg[outlier_idx], y_pred_reg[outlier_idx], 
                color='#ef4444', s=50, label='Outliers', zorder=5)
axes[0].plot([50, 800], [50, 800], color='#22c55e', linestyle='--', linewidth=1.5, label='Perfect')
axes[0].set_xlabel('Actual Price ($K)')
axes[0].set_ylabel('Predicted Price ($K)')
axes[0].set_title('Actual vs Predicted', fontweight='bold', color='#e6e8f4')
axes[0].legend(framealpha=0.3)
axes[0].grid(True, alpha=0.3)

# Residuals
residuals = y_pred_reg - y_true_reg
axes[1].scatter(y_pred_reg, residuals, alpha=0.5, color='#a855f7', s=20)
axes[1].axhline(y=0, color='#22c55e', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Predicted Price ($K)')
axes[1].set_ylabel('Residual ($K)')
axes[1].set_title('Residual Plot', fontweight='bold', color='#e6e8f4')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5. LLM Evaluation: Build a Simple Judge

Build an LLM-as-Judge evaluator. This uses mock responses for demonstration; in production, connect to Gemini via Vertex AI.

In [ ]:
import json
from typing import Dict, List, Optional

class LLMJudge:
    """LLM-as-Judge for evaluating generative AI outputs.
    
    In production, replace the mock with:
        from vertexai.generative_models import GenerativeModel
        self.model = GenerativeModel('gemini-2.0-pro')
    """
    
    POINTWISE_PROMPT = """You are an expert evaluator. Score this AI response on a 1-5 scale for each criterion.

Question: {question}
Response: {response}
Reference (if available): {reference}

Rate each criterion (1=terrible, 5=excellent):
- Relevance: Does it answer the question?
- Accuracy: Is the information correct?
- Completeness: Does it cover all aspects?
- Clarity: Is it well-written and easy to understand?

Output JSON: {{"relevance": X, "accuracy": X, "completeness": X, "clarity": X}}"""
    
    PAIRWISE_PROMPT = """Compare these two responses to the same question.

Question: {question}
Response A: {response_a}
Response B: {response_b}

Which response is better? Output JSON: {{"winner": "A" or "B" or "tie", "reason": "..."}}"""
    
    def __init__(self, model_name: str = "gemini-2.0-pro"):
        self.model_name = model_name
        # self.model = GenerativeModel(model_name)  # Uncomment for production
    
    def _call_model(self, prompt: str) -> str:
        """Call the judge model. Replace with actual API call in production."""
        # In production: return self.model.generate_content(prompt).text
        # Mock: simulate scoring based on response length and keywords
        return ""
    
    def pointwise_evaluate(self, question: str, response: str, 
                           reference: str = "N/A") -> Dict[str, float]:
        """Score a single response on multiple criteria."""
        prompt = self.POINTWISE_PROMPT.format(
            question=question, response=response, reference=reference
        )
        # Mock scoring logic (in production, parse LLM output)
        base_score = min(5, max(1, len(response) / 50 + 2))
        return {
            "relevance": round(min(5, base_score + 0.5), 1),
            "accuracy": round(min(5, base_score), 1),
            "completeness": round(min(5, base_score - 0.3), 1),
            "clarity": round(min(5, base_score + 0.2), 1),
            "overall": round(min(5, base_score + 0.1), 1),
        }
    
    def pairwise_evaluate(self, question: str, response_a: str, response_b: str) -> Dict:
        """Compare two responses head-to-head."""
        winner = "A" if len(response_a) > len(response_b) else "B"
        return {"winner": winner, "reason": f"Response {winner} is more detailed."}
    
    def evaluate_dataset(self, dataset: List[Dict]) -> pd.DataFrame:
        """Evaluate a full dataset and return scores."""
        results = []
        for item in dataset:
            scores = self.pointwise_evaluate(
                question=item["question"],
                response=item["response"],
                reference=item.get("reference", "N/A"),
            )
            results.append({"question": item["question"][:50], **scores})
        return pd.DataFrame(results)


# Demo: Evaluate sample responses
judge = LLMJudge()

eval_dataset = [
    {
        "question": "What is Vertex AI?",
        "response": "Vertex AI is Google Cloud's unified machine learning platform that provides tools for the entire ML workflow including data preparation, model training with AutoML or custom code, model evaluation, deployment to endpoints, and ongoing monitoring. It supports both traditional ML and generative AI workloads.",
        "reference": "Vertex AI is Google Cloud's ML platform for building, deploying, and managing ML models."
    },
    {
        "question": "How do you prevent overfitting?",
        "response": "Use regularization.",
        "reference": "Techniques include regularization (L1/L2), dropout, early stopping, data augmentation, cross-validation, and using more training data."
    },
    {
        "question": "Explain the bias-variance tradeoff.",
        "response": "The bias-variance tradeoff is a fundamental concept in ML. High bias means the model is too simple (underfitting) and misses patterns. High variance means the model is too complex (overfitting) and captures noise. The goal is to find the sweet spot that minimizes total error (bias^2 + variance + irreducible error). Techniques like cross-validation and regularization help navigate this tradeoff.",
        "reference": "Tradeoff between model simplicity (bias) and sensitivity to training data (variance)."
    },
    {
        "question": "What is a confusion matrix?",
        "response": "A confusion matrix is a table showing TP, FP, TN, FN counts for a classifier. It's the basis for precision, recall, and F1.",
        "reference": "A table of predicted vs actual class labels showing true positives, false positives, true negatives, and false negatives."
    },
]

# Pointwise evaluation
print("=== Pointwise Evaluation Results ===")
df_eval = judge.evaluate_dataset(eval_dataset)
print(df_eval.to_string(index=False))

print(f"\nAverage Overall Score: {df_eval['overall'].mean():.2f}/5.00")

# Pairwise evaluation
print("\n=== Pairwise Comparison ===")
result = judge.pairwise_evaluate(
    question="What is Vertex AI?",
    response_a=eval_dataset[0]["response"],
    response_b="Vertex AI is a Google Cloud service for ML.",
)
print(f"Winner: Response {result['winner']}")
print(f"Reason: {result['reason']}")

---
## 6. Vertex AI Model Evaluation SDK

Shows how to use the Vertex AI SDK to access model evaluation results for AutoML models.

In [ ]:
# Vertex AI Model Evaluation SDK usage
# *** REQUIRES GCP PROJECT - shown as reference code ***

# from google.cloud import aiplatform
# aiplatform.init(project="your-project", location="us-central1")
#
# # Get model evaluation for an AutoML model
# model = aiplatform.Model("projects/your-project/locations/us-central1/models/MODEL_ID")
#
# # List all evaluations
# evaluations = model.list_model_evaluations()
# for evaluation in evaluations:
#     print(f"Evaluation: {evaluation.name}")
#     metrics = evaluation.metrics
#     
#     # Classification metrics
#     if hasattr(metrics, 'auRoc'):
#         print(f"  ROC-AUC: {metrics['auRoc']:.4f}")
#         print(f"  Log Loss: {metrics['logLoss']:.4f}")
#     
#     # Get evaluation slices (for fairness analysis)
#     slices = evaluation.list_model_evaluation_slices()
#     for s in slices:
#         print(f"  Slice: {s.slice_.dimension} = {s.slice_.value}")
#         print(f"    Metrics: {s.metrics}")
#
# # For Gen AI Evaluation Service:
# from vertexai.evaluation import EvalTask
#
# eval_task = EvalTask(
#     dataset=[{"prompt": "...", "reference": "...", "response": "..."}],
#     metrics=["fluency", "coherence", "groundedness", "safety"],
# )
# result = eval_task.evaluate(model=GenerativeModel("gemini-2.0-flash"))
# print(result.summary_metrics)

print("Vertex AI Model Evaluation SDK reference code.")
print("Uncomment and run on GCP with a real model ID.")
print("\nKey SDK methods:")
print("  - model.list_model_evaluations()    -> Get evaluation metrics")
print("  - evaluation.list_model_evaluation_slices() -> Slice-based metrics")
print("  - EvalTask.evaluate()               -> Gen AI evaluation")

---
## 7. Slice-Based Evaluation: Evaluate on Subgroups

Check if the model performs fairly across different demographic or business segments.

In [ ]:
# Create a synthetic dataset with subgroup attributes
np.random.seed(42)
n = 2000

# Generate data with different performance across groups
data = pd.DataFrame({
    'gender': np.random.choice(['Male', 'Female', 'Non-binary'], n, p=[0.45, 0.45, 0.1]),
    'age_group': np.random.choice(['18-30', '31-50', '51+'], n, p=[0.3, 0.45, 0.25]),
    'region': np.random.choice(['Urban', 'Suburban', 'Rural'], n, p=[0.4, 0.35, 0.25]),
})

# True labels
data['y_true'] = np.random.binomial(1, 0.3, n)

# Simulate a biased model: lower accuracy for some groups
def biased_predict(row):
    base_accuracy = 0.85
    if row['age_group'] == '51+':
        base_accuracy -= 0.15  # Worse for older adults
    if row['region'] == 'Rural':
        base_accuracy -= 0.10  # Worse for rural
    
    if np.random.random() < base_accuracy:
        return row['y_true']  # Correct prediction
    else:
        return 1 - row['y_true']  # Wrong prediction

data['y_pred'] = data.apply(biased_predict, axis=1)

# Slice-based evaluation
print("=== Slice-Based Evaluation ===")
print(f"{'Slice':<25} {'N':>6} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 75)

# Overall
print(f"{'OVERALL':<25} {n:>6} {accuracy_score(data['y_true'], data['y_pred']):>10.4f} "
      f"{precision_score(data['y_true'], data['y_pred']):>10.4f} "
      f"{recall_score(data['y_true'], data['y_pred']):>10.4f} "
      f"{f1_score(data['y_true'], data['y_pred']):>10.4f}")
print("-" * 75)

# By slices
for col in ['gender', 'age_group', 'region']:
    for value in sorted(data[col].unique()):
        mask = data[col] == value
        subset = data[mask]
        n_sub = len(subset)
        acc = accuracy_score(subset['y_true'], subset['y_pred'])
        prec = precision_score(subset['y_true'], subset['y_pred'], zero_division=0)
        rec = recall_score(subset['y_true'], subset['y_pred'], zero_division=0)
        f1 = f1_score(subset['y_true'], subset['y_pred'], zero_division=0)
        
        flag = " *** BIAS" if acc < 0.75 else ""
        print(f"{col}={value:<15} {n_sub:>6} {acc:>10.4f} {prec:>10.4f} {rec:>10.4f} {f1:>10.4f}{flag}")
    print()

In [ ]:
# Visualize fairness: accuracy by subgroup
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, col in enumerate(['gender', 'age_group', 'region']):
    groups = sorted(data[col].unique())
    accuracies = [accuracy_score(data[data[col]==g]['y_true'], data[data[col]==g]['y_pred']) for g in groups]
    
    colors = ['#3b82f6' if a >= 0.80 else '#f59e0b' if a >= 0.75 else '#ef4444' for a in accuracies]
    bars = axes[idx].bar(groups, accuracies, color=colors, alpha=0.8, edgecolor='#232840')
    axes[idx].axhline(y=0.80, color='#22c55e', linestyle='--', linewidth=1, label='Target (0.80)')
    axes[idx].set_title(f'Accuracy by {col}', fontweight='bold', color='#e6e8f4')
    axes[idx].set_ylim([0.5, 1.0])
    axes[idx].legend(framealpha=0.3, fontsize=9)
    axes[idx].grid(True, alpha=0.2, axis='y')
    
    for bar, acc in zip(bars, accuracies):
        axes[idx].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                       f'{acc:.3f}', ha='center', va='bottom', fontsize=10, color='#e6e8f4')

plt.suptitle('Fairness Analysis: Model Performance by Subgroup', 
             fontweight='bold', color='#e6e8f4', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Red bars indicate subgroups with accuracy below 0.75 - potential bias!")
print("Action: Investigate why the model performs worse for age 51+ and Rural regions.")

---
## 8. A/B Test Significance Calculator

Calculate whether the difference between two model variants is statistically significant.

In [ ]:
class ABTestCalculator:
    """Calculate statistical significance for A/B tests.
    
    Use this to determine if a new model (challenger) is significantly
    better than the current model (champion) in production.
    """
    
    @staticmethod
    def proportion_test(successes_a: int, total_a: int,
                       successes_b: int, total_b: int,
                       alpha: float = 0.05) -> Dict:
        """Two-proportion z-test for conversion rates.
        
        Args:
            successes_a: Number of successes in group A (champion)
            total_a: Total samples in group A
            successes_b: Number of successes in group B (challenger)
            total_b: Total samples in group B
            alpha: Significance level (default 0.05)
        """
        p_a = successes_a / total_a
        p_b = successes_b / total_b
        p_pool = (successes_a + successes_b) / (total_a + total_b)
        
        se = np.sqrt(p_pool * (1 - p_pool) * (1/total_a + 1/total_b))
        z_stat = (p_b - p_a) / se if se > 0 else 0
        p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))  # Two-tailed
        
        ci_diff = 1.96 * np.sqrt(p_a*(1-p_a)/total_a + p_b*(1-p_b)/total_b)
        
        return {
            "champion_rate": p_a,
            "challenger_rate": p_b,
            "absolute_diff": p_b - p_a,
            "relative_diff": (p_b - p_a) / p_a * 100 if p_a > 0 else float('inf'),
            "z_statistic": z_stat,
            "p_value": p_value,
            "significant": p_value < alpha,
            "confidence_interval": (p_b - p_a - ci_diff, p_b - p_a + ci_diff),
            "alpha": alpha,
        }
    
    @staticmethod
    def minimum_sample_size(baseline_rate: float, min_detectable_effect: float,
                            alpha: float = 0.05, power: float = 0.80) -> int:
        """Calculate minimum sample size per variant.
        
        Args:
            baseline_rate: Current conversion rate (e.g., 0.10 for 10%)
            min_detectable_effect: Minimum relative change to detect (e.g., 0.05 for 5%)
            alpha: Significance level
            power: Statistical power (1 - Type II error rate)
        """
        p1 = baseline_rate
        p2 = baseline_rate * (1 + min_detectable_effect)
        
        z_alpha = stats.norm.ppf(1 - alpha/2)
        z_beta = stats.norm.ppf(power)
        
        n = ((z_alpha + z_beta)**2 * (p1*(1-p1) + p2*(1-p2))) / (p2 - p1)**2
        return int(np.ceil(n))


calc = ABTestCalculator()

# Example 1: Compare model accuracy rates
print("=== A/B Test: Model Accuracy Comparison ===")
print("Champion (Model A): 850 correct out of 1000")
print("Challenger (Model B): 880 correct out of 1000")

result = calc.proportion_test(
    successes_a=850, total_a=1000,
    successes_b=880, total_b=1000,
)

print(f"\nChampion accuracy:   {result['champion_rate']:.4f}")
print(f"Challenger accuracy: {result['challenger_rate']:.4f}")
print(f"Absolute diff:       {result['absolute_diff']:+.4f}")
print(f"Relative diff:       {result['relative_diff']:+.2f}%")
print(f"Z-statistic:         {result['z_statistic']:.4f}")
print(f"P-value:             {result['p_value']:.6f}")
print(f"95% CI of diff:      ({result['confidence_interval'][0]:.4f}, {result['confidence_interval'][1]:.4f})")
print(f"\nStatistically significant (p < 0.05)? {'YES - Deploy challenger!' if result['significant'] else 'NO - Need more data.'}")

# Example 2: Calculate required sample size
print("\n\n=== Sample Size Calculator ===")
scenarios = [
    (0.10, 0.05, "10% baseline, detect 5% relative change"),
    (0.10, 0.10, "10% baseline, detect 10% relative change"),
    (0.50, 0.05, "50% baseline, detect 5% relative change"),
    (0.02, 0.10, "2% baseline, detect 10% relative change"),
]

print(f"{'Scenario':<45} {'Sample Size/Variant':>20}")
print("-" * 68)
for baseline, mde, desc in scenarios:
    n = calc.minimum_sample_size(baseline, mde)
    print(f"{desc:<45} {n:>20,}")

In [ ]:
# Visualize: P-value over time as more data comes in
np.random.seed(42)

# Simulate an A/B test running over time
champion_rate = 0.85
challenger_rate = 0.88  # True rate is 3% better

sample_sizes = range(50, 2001, 50)
p_values = []

for n in sample_sizes:
    a_successes = np.random.binomial(n, champion_rate)
    b_successes = np.random.binomial(n, challenger_rate)
    result = calc.proportion_test(a_successes, n, b_successes, n)
    p_values.append(result['p_value'])

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(list(sample_sizes), p_values, color='#3b82f6', linewidth=2)
ax.axhline(y=0.05, color='#ef4444', linestyle='--', linewidth=1.5, label='Significance threshold (p=0.05)')
ax.fill_between(list(sample_sizes), p_values, 0.05, 
                where=[p < 0.05 for p in p_values], alpha=0.2, color='#22c55e', label='Significant')
ax.set_xlabel('Sample Size per Variant')
ax.set_ylabel('P-value')
ax.set_title('A/B Test: P-value Convergence Over Time', fontweight='bold', color='#e6e8f4', fontsize=14)
ax.set_ylim([0, 1])
ax.legend(framealpha=0.3)
ax.grid(True, alpha=0.3)

# Find first significant point
first_sig = next((i for i, p in enumerate(p_values) if p < 0.05), None)
if first_sig is not None:
    n_sig = list(sample_sizes)[first_sig]
    ax.axvline(x=n_sig, color='#22c55e', linestyle=':', alpha=0.5)
    ax.annotate(f'First significant at n={n_sig}', 
                xy=(n_sig, 0.05), xytext=(n_sig + 200, 0.3),
                arrowprops=dict(arrowstyle='->', color='#22c55e'),
                color='#22c55e', fontsize=11)

plt.tight_layout()
plt.show()

print(f"\nWith a {(challenger_rate - champion_rate)*100:.0f}% absolute improvement:")
if first_sig is not None:
    print(f"  First reached significance at ~{n_sig} samples per variant")
print(f"  Recommended: Run for at least 1000+ samples for stable results")
print(f"  Warning: Do NOT stop early just because you see p < 0.05 (peeking problem)")

---
## Key Takeaways for the Exam

1. **Classification**: Use F1/PR-AUC for imbalanced data, not accuracy. Know precision vs recall tradeoff.
2. **Regression**: RMSE is default. MAE is robust to outliers. MAPE for business communication.
3. **GenAI Evaluation**: BLEU/ROUGE are limited. Use LLM-as-Judge for quality. Know pointwise vs pairwise.
4. **Vertex AI**: AutoML provides automatic evaluation. Gen AI Evaluation Service for LLM assessment.
5. **Fairness**: Always do slice-based evaluation. Look for performance disparities across subgroups.
6. **A/B Testing**: Calculate sample size before starting. Don't peek. Use proper statistical tests.
7. **Continuous Evaluation**: Monitor deployed models over time. Set up retraining triggers.
8. **Golden Datasets**: Curate high-quality evaluation sets. Prevent contamination with training data.